In [ ]:
import scanpy as sc
import scvelo as scv
import anndata as ad
import numpy as np

# --------------------------
# 0. Settings (REPRODUCIBILITY)
# --------------------------

from pathlib import Path

cwd = Path.cwd()
ANALYSIS_DIR = (
    cwd
    if cwd.name == "06_b_cell_igvf"
    else Path("06_b_cell_igvf")
    if Path("06_b_cell_igvf").exists()
    else Path("..").resolve()
    if cwd.name == "notebooks"
    else Path("../..").resolve()
)
DATA_DIR = Path("data/flowmap_manuscript/b_cell")
if not DATA_DIR.exists():
    DATA_DIR = ANALYSIS_DIR.parent / "data" / "flowmap_manuscript" / "b_cell"
RESULTS_DIR = Path("data/flowmap_manuscript/b_cell_results")
if not RESULTS_DIR.exists():
    RESULTS_DIR = ANALYSIS_DIR.parent / "data" / "flowmap_manuscript" / "b_cell_results"
FIGURE_DIR = ANALYSIS_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
UTILS_DIR = ANALYSIS_DIR / "utils"

sc.settings.verbosity = 3
scv.settings.verbosity = 3

np.random.seed(0)

In [ ]:
# --------------------------
# 1. Load
# --------------------------
adata = ad.read_h5ad(DATA_DIR / "IGVFFI3928IUMP.h5ad")
adata

In [ ]:
# --------------------------
# 2. Fast QC (NO calculate_qc)
# --------------------------

# basic filters only (fast)
sc.pp.filter_cells(adata, min_counts=1000)
sc.pp.filter_cells(adata, min_genes=500)

sc.pp.filter_genes(adata, min_cells=20)

# optional: very cheap mito filter (vectorized, no heavy metrics)
mt_mask = adata.var_names.str.startswith("MT-")
pct_mt = (
    np.array(adata[:, mt_mask].X.sum(axis=1)).flatten() /
    np.array(adata.X.sum(axis=1)).flatten()
)

adata = adata[pct_mt < 0.2].copy()
adata

In [ ]:
# --------------------------
# 3. Set layers (CRITICAL for scVelo)
# --------------------------
adata.layers["spliced"] = adata.layers["mature"]
adata.layers["unspliced"] = adata.layers["nascent"]

# --------------------------
# 4. scVelo preprocessing (DO NOT use scanpy normalize here)
# --------------------------
scv.pp.filter_and_normalize(
    adata,
    min_shared_counts=20,
    n_top_genes=2000
)

In [ ]:
# --------------------------
# 5. Moments (graph construction)
# --------------------------
scv.pp.moments(
    adata,
    n_pcs=30,
    n_neighbors=30
)

In [ ]:
scv.tl.velocity(adata, mode="stochastic")
scv.tl.velocity_graph(adata)

In [ ]:
sc.tl.umap(adata, random_state=0)
scv.pl.velocity_embedding_stream(
    adata,
    basis="umap"
)

In [ ]:
import mygene
import pandas as pd
import numpy as np

def clean_gene_names(adata):
    mg = mygene.MyGeneInfo()
    
    # 1. Prepare query list (clean versioning like .1, .2)
    original_ids = adata.var_names.tolist()
    clean_ids = [str(g).split('.')[0] for g in original_ids]
    
    # 2. Query as a list of dicts (more robust than the dataframe output)
    print(f"Querying {len(clean_ids)} genes...")
    results = mg.querymany(
        clean_ids, 
        scopes="ensembl.gene", 
        fields="symbol", 
        species="human", 
        as_dataframe=False, # Use list of dicts to avoid index/KeyErrors
        verbose=False
    )
    
    # 3. Build a strict mapping: Only add if 'symbol' actually exists
    symbol_map = {}
    for item in results:
        query_id = item.get('query')
        symbol = item.get('symbol')
        if query_id and symbol and str(symbol).lower() != 'nan':
            symbol_map[query_id] = str(symbol)

    # 4. Final assignment: Map it or Keep it
    new_names = []
    for i, orig in enumerate(original_ids):
        clean = clean_ids[i]
        # Priority: 1. Found Symbol, 2. Cleaned ID, 3. Original string
        final_name = symbol_map.get(clean, clean if str(clean).lower() != 'nan' else orig)
        new_names.append(final_name)
    
    # 5. Update AnnData
    adata.var['original_id'] = original_ids # Keep a backup just in case
    adata.var_names = new_names
    adata.var_names_make_unique()
    
    print(f"Mapping complete. Unique names assigned.")
    return adata

# Execution
adata = clean_gene_names(adata)
adata

In [ ]:
scv.tl.recover_dynamics(adata)
scv.tl.latent_time(adata)

In [ ]:
scv.pl.velocity_embedding_stream(
    adata,
    basis="umap",
    color="velocity_pseudotime"
)
adata.write(DATA_DIR / "bcell_velocity_standard.h5ad")

In [ ]:
# import anndata as ad
# import scanpy as sc
# import numpy as np
# import pandas as pd

# adata = ad.read_h5ad(DATA_DIR / "bcell_velocity_standard.h5ad")
# adata_raw = ad.read_h5ad(DATA_DIR / "IGVFFI3928IUMP.h5ad")

# # same basic filtering (cells only, DO NOT subset genes aggressively)
# sc.pp.filter_cells(adata_raw, min_counts=1000)
# sc.pp.filter_cells(adata_raw, min_genes=500)

# # normalize + log (same as tutorial)
# sc.pp.normalize_total(adata_raw, target_sum=1e4)
# sc.pp.log1p(adata_raw)

# # ============================================================
# # Convert Ensembl → gene symbols (for adata_raw)
# # ============================================================
# import mygene

# mg = mygene.MyGeneInfo()

# original_ids = adata_raw.var_names.tolist()
# clean_ids = [str(g).split('.')[0] for g in original_ids]

# print("Mapping genes for cell cycle scoring...")

# res = mg.querymany(
#     clean_ids,
#     scopes="ensembl.gene",
#     fields="symbol",
#     species="human",
#     as_dataframe=False,
#     verbose=False
# )

# symbol_map = {}
# for item in res:
#     q = item.get("query")
#     s = item.get("symbol")
#     if q and s and str(s).lower() != "nan":
#         symbol_map[q] = str(s)

# # assign symbols (fallback to Ensembl if missing)
# new_names = []
# for i, orig in enumerate(original_ids):
#     clean = clean_ids[i]
#     new_names.append(symbol_map.get(clean, clean))

# adata_raw.var["original_id"] = original_ids
# adata_raw.var_names = new_names
# adata_raw.var_names_make_unique()

# print("Gene mapping done.")

# # ============================================================
# # Cell cycle gene list (Scanpy tutorial)
# # ============================================================
# import urllib.request

# url = "https://raw.githubusercontent.com/theislab/scanpy_usage/master/180209_cell_cycle/data/regev_lab_cell_cycle_genes.txt"
# urllib.request.urlretrieve(url, UTILS_DIR / "cell_cycle_genes.txt")

# cell_cycle_genes = [x.strip() for x in open(UTILS_DIR / "cell_cycle_genes.txt")]

# s_genes = cell_cycle_genes[:43]
# g2m_genes = cell_cycle_genes[43:]

# # filter to genes that exist
# s_genes = [g for g in s_genes if g in adata_raw.var_names]
# g2m_genes = [g for g in g2m_genes if g in adata_raw.var_names]


# # ============================================================
# # Compute cell cycle
# # ============================================================
# sc.tl.score_genes_cell_cycle(
#     adata_raw,
#     s_genes=s_genes,
#     g2m_genes=g2m_genes
# )


# # ============================================================
# # Transfer scores to your processed adata
# # ============================================================
# common_cells = adata.obs_names.intersection(adata_raw.obs_names)

# adata.obs.loc[common_cells, "S_score"]   = adata_raw.obs.loc[common_cells, "S_score"]
# adata.obs.loc[common_cells, "G2M_score"] = adata_raw.obs.loc[common_cells, "G2M_score"]
# adata.obs.loc[common_cells, "phase"]     = adata_raw.obs.loc[common_cells, "phase"]

# # optional composite
# adata.obs["cycle_score"] = (
#     adata.obs["S_score"] + adata.obs["G2M_score"]
# )

# print("Cell cycle scores added.")

# # ============================================================
# # Add p21 / p27 expression (G0 markers)
# # ============================================================
# genes = {
#     "p21": "CDKN1A",
#     "p27": "CDKN1B"
# }

# for key, gene in genes.items():
    
#     if gene not in adata_raw.var_names:
#         print(f"{gene} not found in adata_raw")
#         continue
    
#     x = adata_raw[:, gene].X
    
#     # handle sparse
#     if hasattr(x, "toarray"):
#         x = x.toarray().flatten()
#     else:
#         x = np.array(x).flatten()
    
#     # align to processed adata
#     aligned = np.full(adata.n_obs, np.nan)
    
#     raw_idx = adata_raw.obs_names.get_indexer(common_cells)
#     proc_idx = adata.obs_names.get_indexer(common_cells)
    
#     aligned[proc_idx] = x[raw_idx]
    
#     adata.obs[key] = aligned

# print("p21 / p27 added to adata.obs")

# # adata.write(DATA_DIR / "bcell_velocity_standard.h5ad")

In [ ]:
import scvelo as scv

# Optional: re-compute neighbors if unsure
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)

# Leiden clustering
sc.tl.leiden(adata, resolution=0.8)  # adjust later if needed

scv.pl.velocity_embedding_stream(
    adata,
    basis="umap",
    color="leiden"
)

In [ ]:
markers = {
    "Naive": ["GRASP", "ZNF331", "IRS2", "LIX1-AS1", "NR4A2", "KCNH8", "DUSP1", "RASGEF1B", "FOSB", "ZBTB10"],
    "ActB": ["ITGA1", "JAZF1", "SMIM14", "AC083837.1", "ST6GALNAC3", "IL7", "NIBAN3"],
    "preGCBC": ["KIAA1549L", "CFI", "HOMER2", "EEPD1", "PALLD", "SLC37A3", "FCER2"],
    "prePB": ["IL2RB", "IL2RA", "CD226", "DUSP4", "ACSL4", "CLIC5", "IL12RB2"],
    "PB": ["CFAP54", "AL591518.1", "ACOXL", "FNDC3B", "AC016074.2", "NUGGC", "RASSF6", "ZNF215"]
}

markers_filtered = {}

for key, gene_list in markers.items():
    genes_present = [g for g in gene_list if g in adata.var_names]
    if len(genes_present) > 0:
        markers_filtered[key] = genes_present

markers_filtered

In [ ]:
adata

In [ ]:
# adata_raw.obs["leiden"] = adata.obs["leiden"]
sc.pl.dotplot(
    adata,
    markers_filtered,
    groupby='leiden',
    standard_scale='var'
)

In [ ]:
# mapping dict
cluster_map = {
    "0": "ActB",
    "1": "PB",
    "2": "prePB",
    "3": "ActB",
    "4": "ActB",
    "5": "preGCBC",
    "6": "Naive",
    "7": "Naive",
    "8": "prePB",
    "9": "Naive",
    "10": "Naive",
    "11": "Naive",
}

# assign new labels
adata.obs["celltype"] = adata.obs["leiden"].map(cluster_map)
adata.write(DATA_DIR / "bcell_velocity_standard.h5ad")